# Training curves

The training and validation loss of one or more runs, with their checkpoints, to see when a model stops improving on unseen data (overfitting) and which checkpoint to evaluate.

Every run below `EXPERIMENTS` is found, of both kinds:
- **model zoo** (`scripts/train_model.py`): `history.json`, checkpoints `checkpoint_<step>/`, the best one by the validation loss (`keep_best`);
- **SO3PriorTrainer** (the old flow trainer, e.g. `prior_c30/id_9030`): the curves from its TensorBoard events, checkpoints `model_ckpt_pdf_estimator_<step - 1>.pt`.

What is plotted depends on the model:
- **flows** (`so3_flow`): the loss is the negative log-likelihood, so the curves are shown as **log-likelihood** (higher is better). The gap between training and validation is the generalization gap in nats;
- **RSGM**: the denoising score matching loss (lower is better). Its validation is of the EMA weights with fixed noise, and the loss is no likelihood: compare runs of RSGM with each other, not with the flows.

Models of the two kinds go to separate columns. In the plots, dotted vertical lines mark the saved checkpoints (in the colour of the run), squares mark the validation at a checkpoint, a star marks the best of the saved checkpoints (by its validation; runs from before `keep_best` have none), and a diamond marks the end of the run (the final weights). With "clip outliers", validations outside the view (broken ones, as of RSGM r2t12) are shown as triangles at the edge.

In [ ]:
import json
import re
from functools import lru_cache
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from omegaconf import OmegaConf

from hp.training import best_checkpoint, list_checkpoints

# ---- parameters ----------------------------------------------------------------------
EXPERIMENTS = "~/data/experiments"                     # runs are <EXPERIMENTS>/<name>/<run> folders
DEFAULT = ["prior_full/run_0001", "prior_c30/id_9030"]  # selected at the start (flow full and flow c30)
SMOOTH = 200                                            # steps of the moving average of the training loss
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
NLL_MODELS = ("so3_flow",)  # models whose loss is the negative log-likelihood
LEGACY_CHECKPOINT = re.compile(r"model_ckpt_pdf_estimator_(\d+)\.pt")


def find_runs(root):
    # {"<name>/<run>": folder} for every run of the model zoo or of SO3PriorTrainer below root.
    root = expand(root)
    runs = {}
    for p in sorted(root.glob("*/*")):
        if (p / "history.json").is_file() and (p / "model.json").is_file():
            runs[str(p.relative_to(root))] = p
        elif (p / "model_ckpt_pdf_estimator.pt").is_file() and (p / "config.yaml").is_file() and (p / "tensorboard").is_dir():
            runs[str(p.relative_to(root))] = p
    return runs


def _tensorboard_scalars(folder):
    # {tag: (steps, values)} of all event files below folder.
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

    scalars = {}
    for f in sorted(folder.glob("**/events.out.tfevents.*")):
        events = EventAccumulator(str(f), size_guidance={"scalars": 0})
        events.Reload()
        for tag in events.Tags()["scalars"]:
            s = events.Scalars(tag)
            scalars[tag] = (np.array([e.step for e in s]), np.array([e.value for e in s], dtype=float))
    return scalars


@lru_cache(maxsize=None)
def load_run(name):
    # The curves and checkpoints of a run: steps count optimizer steps made (the first step is 1).
    folder = RUNS[name]
    if (folder / "history.json").is_file():  # model zoo
        h = json.loads((folder / "history.json").read_text())
        model = json.loads((folder / "model.json").read_text())["build_args"]["name"]
        cfg = OmegaConf.load(folder / "config.yaml")
        train = np.asarray(h["loss"], dtype=float)
        best = best_checkpoint(folder)
        state = folder / "train_state.pt"
        sizes = torch.load(state, map_location="cpu", weights_only=True).get("data_sizes") if state.is_file() else None
        run = dict(kind="model zoo", model=model, train_steps=np.arange(1, len(train) + 1), train=train,
                   valid_steps=np.asarray(h["valid_iteration"]), valid=np.asarray(h["valid_loss"], dtype=float),
                   checkpoints=[int(p.name[-6:]) for p in list_checkpoints(folder)], best=int(best.name[-6:]) if best else None,
                   max_iterations=cfg.train.max_iterations, batch=cfg.train.batch_size, frames=sizes["train"] if sizes else None)
    else:  # SO3PriorTrainer: its TensorBoard steps start at 0
        scalars = _tensorboard_scalars(folder / "tensorboard")
        cfg = OmegaConf.load(folder / "config.yaml")
        (ts, tv), (vs, vv) = scalars["Loss/train"], scalars["Loss/val"]
        checkpoints = sorted(int(m.group(1)) + 1 for f in folder.iterdir() if (m := LEGACY_CHECKPOINT.fullmatch(f.name)))
        run = dict(kind="SO3PriorTrainer", model="so3_flow", train_steps=ts + 1, train=tv, valid_steps=vs + 1, valid=vv,
                   checkpoints=checkpoints, best=None, max_iterations=cfg.max_iterations, batch=cfg.batch_size, frames=None)
    run["nll"] = run["model"] in NLL_MODELS
    run["quantity"] = "log-likelihood (higher is better)" if run["nll"] else f"{run['model']} loss (lower is better)"
    return run


def smoothed(steps, values, n):
    n = max(1, min(n, len(values)))
    return steps[n - 1:], np.convolve(values, np.ones(n) / n, mode="valid")


RUNS = find_runs(EXPERIMENTS)
print(f"{len(RUNS)} runs below {EXPERIMENTS}:")
for name in RUNS:
    r = load_run(name)
    print(f"  {name:36s} {r['model']:9s} {r['kind']:16s} {len(r['train']):7,} steps, {len(r['checkpoints'])} checkpoints")

## Summary

Per run: the validation at its best point and at the end, and the checkpoints that are on disk. For flows in nats of log-likelihood, for RSGM in its loss. "gap": training minus validation log-likelihood at the end (flows; the training value is the moving average of the last `SMOOTH` steps).

In [ ]:
def summary(names):
    print(f"{'run':36s} {'model':9s} {'best valid':>17s} {'end valid':>10s} {'end train':>10s} {'gap':>7s}  checkpoints (* best)")
    for name in names:
        r = load_run(name)
        sign = -1 if r["nll"] else 1
        v = sign * r["valid"]
        i = int(np.argmax(v) if r["nll"] else np.argmin(v))
        train_end = sign * r["train"][-SMOOTH:].mean()
        gap = f"{train_end - v[-1]:7.1f}" if r["nll"] else "      -"
        marks = " ".join(f"{c // 1000}k{'*' if c == r['best'] else ''}" if c % 1000 == 0 else f"{c}{'*' if c == r['best'] else ''}"
                         for c in r["checkpoints"])
        print(f"{name:36.36s} {r['model']:9s} {v[i]:9.2f} @{r['valid_steps'][i]:6,} {v[-1]:10.2f} {train_end:10.2f} {gap}  {marks}")


summary([n for n in RUNS])

## The curves

Top: training (moving average over `SMOOTH` steps; the raw values faint) and validation, with the checkpoints. Bottom: the generalization gap at every validation, training minus validation log-likelihood for flows (validation minus training loss for other models). A growing gap with a flat or falling validation curve is overfitting: the best checkpoint is then before the end.

In [ ]:
def robust_range(values, k=6.0):
    # [min, max] of the values within k robust standard deviations (1.4826 MAD) of their median: what a curve
    # shows without its first steps and without broken values (e.g. the loss spikes and validations of RSGM r2t12).
    deviation = np.abs(values - np.median(values))
    scale = 1.4826 * np.median(deviation)
    kept = values[deviation <= k * scale] if scale > 0 else values
    return kept.min(), kept.max()


def plot(names, log_x=False, clip=True, smooth=SMOOTH):
    runs = {n: load_run(n) for n in names}
    colors = {n: f"C{k % 10}" for k, n in enumerate(names)}
    quantities = list(dict.fromkeys(r["quantity"] for r in runs.values()))
    fig, axes = plt.subplots(2, len(quantities), figsize=(8 * len(quantities), 7.5), squeeze=False,
                             gridspec_kw={"height_ratios": [3, 1.2]}, sharex="col")
    for col, quantity in enumerate(quantities):
        ax, gap_ax = axes[0, col], axes[1, col]
        bands, valids, gaps = [], [], []  # for the limits: robust ranges of the curves, and the validations to mark if outside
        for name, r in runs.items():
            if r["quantity"] != quantity:
                continue
            c, sign = colors[name], (-1 if r["nll"] else 1)
            train, valid = sign * r["train"], sign * r["valid"]
            ax.plot(r["train_steps"], train, color=c, alpha=0.12, lw=0.5)
            ss, sv = smoothed(r["train_steps"], train, smooth)
            ax.plot(ss, sv, color=c, lw=1.6, label=f"{name}: training")
            ax.plot(r["valid_steps"], valid, "o--", color=c, ms=3, lw=1, label=f"{name}: validation")
            bands += [robust_range(sv), robust_range(valid)]
            valids.append((r["valid_steps"], valid, c))
            at = dict(zip(r["valid_steps"].tolist(), valid.tolist()))  # validation at a step, if there is one
            for step in r["checkpoints"]:
                ax.axvline(step, color=c, ls=":", lw=0.9, alpha=0.45)
                if step in at:
                    ax.plot(step, at[step], "s", color=c, ms=6, mfc="none", mew=1.4)
            if r["best"] is not None and r["best"] in at:
                ax.plot(r["best"], at[r["best"]], "*", color=c, ms=16, mec="k", mew=0.7, zorder=5)
                ax.annotate(f"best {r['best']:,}", (r["best"], at[r["best"]]), textcoords="offset points", xytext=(6, 8),
                            color=c, fontsize=8)
            if r["valid_steps"].size and r["valid_steps"][-1] == r["max_iterations"]:
                ax.plot(r["valid_steps"][-1], valid[-1], "D", color=c, ms=7, mec="k", mew=0.7, zorder=5)
            # the gap at every validation, against the smoothed training curve
            train_at_valid = np.interp(r["valid_steps"], ss, sv)
            gap = train_at_valid - valid if r["nll"] else valid - train_at_valid
            gap_ax.plot(r["valid_steps"], gap, "o-", color=c, ms=3, lw=1)
            gaps.append(robust_range(gap))
        if clip and bands:  # robust limits: the start of training and a few broken validations do not squash the rest
            lo, hi = min(b[0] for b in bands), max(b[1] for b in bands)
            pad = 0.08 * (hi - lo)
            lo, hi = lo - pad, hi + pad
            ax.set_ylim(lo, hi)
            g_lo, g_hi = min(min(g[0] for g in gaps), 0), max(g[1] for g in gaps)
            gap_ax.set_ylim(g_lo - 0.08 * (g_hi - g_lo), g_hi + 0.08 * (g_hi - g_lo))
            for steps, valid, c in valids:  # validations outside the view: an arrow at the edge, not silently gone
                for outside, edge, marker in ((valid > hi, hi, "^"), (valid < lo, lo, "v")):
                    if outside.any():  # (an empty unclipped line would confuse tight_layout)
                        ax.plot(steps[outside], np.full(outside.sum(), edge), marker, color=c, ms=7, clip_on=False)
        ax.set_title(quantity)
        ax.set_ylabel(quantity.split(" (")[0])
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="best")
        gap_ax.set_ylabel("gap: train - valid" if "log-likelihood" in quantity else "gap: valid - train")
        gap_ax.set_xlabel("step")
        gap_ax.grid(alpha=0.3)
        gap_ax.axhline(0, color="k", lw=0.6)
        if log_x:
            gap_ax.set_xscale("log")
        group = [r for r in runs.values() if r["quantity"] == quantity]
        if len(group) == 1 and group[0]["frames"]:  # epochs on top for a single run that knows its data
            per_step = group[0]["batch"] / group[0]["frames"]  # epochs per step
            ax.secondary_xaxis("top", functions=(lambda s: s * per_step, lambda e: e / per_step)).set_xlabel("epoch")
    fig.tight_layout()
    plt.show()


plot([n for n in DEFAULT if n in RUNS])

## Choose the runs

Select one or more runs (Cmd/Ctrl-click for several). The plot follows the selection.

In [ ]:
select = widgets.SelectMultiple(options=list(RUNS), value=[n for n in DEFAULT if n in RUNS], rows=min(14, len(RUNS)),
                                description="runs", layout=widgets.Layout(width="430px"))
log_x = widgets.Checkbox(value=False, description="log x")
clip = widgets.Checkbox(value=True, description="clip outliers")
smooth = widgets.IntSlider(value=SMOOTH, min=1, max=2000, step=25, description="smooth")


def show(names, log_x, clip, smooth):
    if not names:
        print("Select at least one run.")
        return
    summary(names)
    plot(list(names), log_x, clip, smooth)


display(widgets.HBox([select, widgets.VBox([log_x, clip, smooth])]),
        widgets.interactive_output(show, dict(names=select, log_x=log_x, clip=clip, smooth=smooth)))